### Högbom 方法


这是 Högbom `CLEAN` 去卷积方法的一个实现示例。若历史 FITS 输入不存在，notebook 会生成一组确定性的合成数据，因此可以独立运行。示例默认单偏振、单频率，并把 PSF 峰值归一化为 1。

伪代码如下：


$\textbf{input: } I^{D}(l,m), \ B(l,m), \ \gamma, \ f_{\textrm{thresh}}, \ N, \ \mathcal{M}$

$\textbf{initialize: } I^{\textrm{model}} \leftarrow 0, I^{\textrm{res}} \leftarrow I^{D}, i \leftarrow 0$

$\textbf{while} \ \max_{(l,m)\in\mathcal{M}}|I^{\textrm{res}}(l,m)| > f_{\textrm{thresh}} \ \textbf{and} \ i < N \ \textbf{do:}$

$\qquad (l_p,m_p) \leftarrow \underset{(l,m)\in\mathcal{M}}{\operatorname{argmax}} |I^{\textrm{res}}(l,m)|$

$\qquad a_i \leftarrow \gamma I^{\textrm{res}}(l_p,m_p)$

$\qquad I^{\textrm{res}}(l,m) \leftarrow I^{\textrm{res}}(l,m) - a_i B(l-l_p,m-m_p)$

$\qquad I^{\textrm{model}}(l,m) \leftarrow I^{\textrm{model}}(l,m) + a_i\delta(l-l_p)\delta(m-m_p)$

$\qquad i \leftarrow i +1$

$\textbf{output: } I^{\textrm{model}}, I^{\textrm{res}}$

这里的 $\mathcal{M}$ 是可选 CLEAN mask；若不提供，就搜索整幅图。用绝对值寻找峰值但保留 $I^{\textrm{res}}(l_p,m_p)$ 的符号，才能同时处理正、负分量。

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

from clean_demo import gaussian_clean_beam, hogbom_clean, load_clean_data, restore_image


In [ ]:
# clean_demo.subtract_psf clips shifted PSFs at image boundaries instead of wrapping them.


In [ ]:
# gaussian_clean_beam estimates a peak-normalized restoring beam from the PSF main lobe.


In [ ]:
# restore_image keeps signed CLEAN components and uses linear convolution.


In [ ]:
# hogbom_clean implements the pseudocode above and also accepts a CLEAN mask.


***

In [ ]:
gain = 0.1
niter = 100
fthresh = 3.0


In [ ]:
dirtyImg, psfImg, dataSource = load_clean_data(
    '../data/fits/deconv/KAT-7_6h60s_dec-30_10MHz_10chans_uniform_n100-dirty.fits',
    '../data/fits/deconv/KAT-7_6h60s_dec-30_10MHz_10chans_uniform_n100-psf.fits',
)
print(f'Data source: {dataSource}')
cleanBeam = gaussian_clean_beam(psfImg)


In [ ]:
cleanResult = hogbom_clean(dirtyImg, psfImg, gain=gain, niter=niter, threshold=fthresh)
skyModel = cleanResult['model']
residImg = cleanResult['residual']
print(f"Iterations: {cleanResult['iterations']}")


In [ ]:
#plot dirty image
fig = plt.figure(figsize=(8,8))
plt.imshow(dirtyImg, origin='lower')
plt.title('Dirty Image')
plt.colorbar()


In [ ]:
#plot residual image
fig = plt.figure(figsize=(8,8))
plt.imshow(residImg, origin='lower')
plt.title('Residual Image')
plt.colorbar()


In [ ]:
#plot restored image
restImg = restore_image(skyModel, residImg, cleanBeam)
fig = plt.figure(figsize=(8,8))
plt.imshow(restImg, origin='lower')
plt.title('Restored Image')
plt.colorbar()
